In [3]:

import pandas as pd
import numpy as np


DATA_PATH = r"C:\Users\srahman3\OneDrive - The University of Texas at El Paso\Desktop\Summer 25\Supplier Selection\final dataset\Supplier_Disruption_LogTable22.csv"

df = pd.read_csv(DATA_PATH)

# Clean supplier names
df["supplier"] = (
    df["supplier"]
    .astype(str)
    .str.strip()
    .str.replace(",", "", regex=False)
    .str.upper()
)


criteria_cols = [
    "kg_score",
    "Em_Score",
    "twitter_alert",
    "weather_risk",
    "economic_risk",
    "geo_risk",
    "tech_risk",
    "downtime_hours",
    "high_disruption_p"
]


df[criteria_cols] = df[criteria_cols].fillna(0)


#  Aggregate to Supplier Level


supplier_df = df.groupby("supplier", as_index=False)[criteria_cols].mean()


# 5️⃣ Normalize (Min-Max)


for col in criteria_cols:
    min_val = supplier_df[col].min()
    max_val = supplier_df[col].max()
    supplier_df[col] = (supplier_df[col] - min_val) / (max_val - min_val + 1e-9)



supplier_df["high_disruption_p"] = 1 - supplier_df["high_disruption_p"]



weights = np.array([
    0.08,   # kg_score
    0.07,   # Em_Score
    0.05,   # twitter_alert
    0.05,   # weather_risk
    0.08,   # economic_risk
    0.05,   # geo_risk
    0.05,   # tech_risk
    0.07,   # downtime_hours
    0.50    # high_disruption_p (dominant)
])

weights = weights / weights.sum()

print("\nAssigned Weights:")
for c, w in zip(criteria_cols, weights):
    print(f"{c}: {round(w,4)}")


# 8️⃣ Compute FAHP Score


supplier_df["FAHP_score"] = np.dot(
    supplier_df[criteria_cols],
    weights
)


# 9️⃣ Rank Suppliers (Higher Score = Safer)


supplier_df = supplier_df.sort_values("FAHP_score", ascending=False)
supplier_df["FAHP_Rank"] = range(1, len(supplier_df) + 1)

print("\nSafety-Oriented FAHP Supplier Ranking:")
print(supplier_df[["supplier", "FAHP_score", "FAHP_Rank"]])


Assigned Weights:
kg_score: 0.08
Em_Score: 0.07
twitter_alert: 0.05
weather_risk: 0.05
economic_risk: 0.08
geo_risk: 0.05
tech_risk: 0.05
downtime_hours: 0.07
high_disruption_p: 0.5

Safety-Oriented FAHP Supplier Ranking:
  supplier  FAHP_score  FAHP_Rank
9      ZSM    0.628007          1
1     DMEC    0.609623          2
5      SCT    0.575949          3
4      KMP    0.519469          4
6     SZHA    0.438317          5
2      GHM    0.397797          6
8       YG    0.349145          7
3      JHH    0.336589          8
0      ARC    0.332867          9
7     YAHO    0.205294         10


In [4]:
import numpy as np
import pandas as pd

# ==========================================================
# 1️⃣ Load Dataset
# ==========================================================

DATA_PATH = r"C:\Users\srahman3\OneDrive - The University of Texas at El Paso\Desktop\Summer 25\Supplier Selection\final dataset\Supplier_Disruption_LogTable22.csv"

df = pd.read_csv(DATA_PATH)

df["supplier"] = (
    df["supplier"]
    .astype(str)
    .str.strip()
    .str.replace(",", "", regex=False)
    .str.upper()
)

criteria_cols = [
    "kg_score",
    "Em_Score",
    "twitter_alert",
    "weather_risk",
    "economic_risk",
    "geo_risk",
    "tech_risk",
    "downtime_hours",
    "high_disruption_p"
]

df[criteria_cols] = df[criteria_cols].fillna(0)

supplier_df = df.groupby("supplier", as_index=False)[criteria_cols].mean()


# Define Triangular Fuzzy Scale


fuzzy_scale = {
    1: (1,1,1),
    3: (2,3,4),
    5: (4,5,6),
    7: (6,7,8),
    9: (8,9,10)
}



n = len(criteria_cols)
pairwise = np.ones((n,n))

# Disruption very strong vs others
for i in range(n-1):
    pairwise[8,i] = 7
    pairwise[i,8] = 1/7

# kg_score moderately strong
pairwise[0,4] = 3
pairwise[4,0] = 1/3

# Others equal
# (you may refine this logically if needed)


# 4️⃣ Convert to Fuzzy Matrix


fuzzy_matrix = []

for i in range(n):
    row = []
    for j in range(n):
        val = pairwise[i,j]
        if val >= 1:
            l,m,u = fuzzy_scale.get(int(val), (1,1,1))
        else:
            inv = int(round(1/val))
            l,m,u = fuzzy_scale.get(inv, (1,1,1))
            l,m,u = (1/u,1/m,1/l)
        row.append((l,m,u))
    fuzzy_matrix.append(row)

fuzzy_matrix = np.array(fuzzy_matrix)



# Step 1: Row sums
row_sums = np.sum(fuzzy_matrix, axis=1)

# Step 2: Total sum
total_sum = np.sum(row_sums, axis=0)

# Step 3: Compute synthetic extent
S = []

for i in range(n):
    l = row_sums[i][0] / total_sum[2]
    m = row_sums[i][1] / total_sum[1]
    u = row_sums[i][2] / total_sum[0]
    S.append((l,m,u))

S = np.array(S)

# Step 4: Defuzzification (Center of Area)
weights = np.array([(l+m+u)/3 for l,m,u in S])
weights = weights / weights.sum()

print("\nFAHP Derived Weights:")
for c, w in zip(criteria_cols, weights):
    print(f"{c}: {round(w,4)}")



for col in criteria_cols:
    min_val = supplier_df[col].min()
    max_val = supplier_df[col].max()
    supplier_df[col] = (supplier_df[col] - min_val) / (max_val - min_val + 1e-9)

# Treat disruption as COST criterion
supplier_df["high_disruption_p"] = 1 - supplier_df["high_disruption_p"]



supplier_df["FAHP_score"] = np.dot(
    supplier_df[criteria_cols],
    weights
)

supplier_df = supplier_df.sort_values("FAHP_score", ascending=False)
supplier_df["FAHP_Rank"] = range(1, len(supplier_df)+1)

print("\nFinal FAHP Supplier Ranking:")
print(supplier_df[["supplier","FAHP_score","FAHP_Rank"]])


FAHP Derived Weights:
kg_score: 0.0822
Em_Score: 0.0657
twitter_alert: 0.0657
weather_risk: 0.0657
economic_risk: 0.0606
geo_risk: 0.0657
tech_risk: 0.0657
downtime_hours: 0.0657
high_disruption_p: 0.463

Final FAHP Supplier Ranking:
  supplier  FAHP_score  FAHP_Rank
9      ZSM    0.591268          1
1     DMEC    0.580323          2
5      SCT    0.553231          3
4      KMP    0.522761          4
6     SZHA    0.449844          5
2      GHM    0.398003          6
8       YG    0.380555          7
3      JHH    0.358885          8
0      ARC    0.340604          9
7     YAHO    0.192489         10
